# Baseline 3 — StyleGAN2-ADA on Batik_Lasem (canonical 50% subset)

**Project:** Deep GANs for Aesthetic-Driven Apparel Pattern Synthesis.

Uses the **official NVIDIA StyleGAN2-ADA (PyTorch)** implementation
(`NVlabs/stylegan2-ada-pytorch`, pinned commit) — not a from-scratch imitation.
Generates **128×128 RGB** Batik_Lasem motifs.

**Comparability (critical):**
* Trains on the **exact same canonical 50% TRAIN split** as DCGAN / cWGAN-GP
  (`data/splits/batik_lasem_50pct_train.csv`), pre-resized to 128×128 with the
  **identical** bicubic+antialias preprocessing, then packed with `dataset_tool.py`.
* The held-out **TEST** split is **never** given to `train.py`.
* StyleGAN2-ADA logs its **own** `fid50k_full` (computed against the *training*
  reals). For a fair cross-model comparison we **also recompute FID/KID against
  our held-out TEST set** using the *same* `batik_gan.metrics` evaluator as the
  other two baselines. Both numbers are reported and clearly labelled.

**Checkpoint/resume:** native — snapshots `network-snapshot-*.pkl`; resume via
`--resume`. Output dir is on Google Drive so runs survive Colab restarts.

> ⚠️ StyleGAN2-ADA compiles custom CUDA ops (`bias_act`, `upfirdn2d`) on first
> use and has stricter environment needs than the other baselines. The setup cell
> installs `ninja` and pins the commit. On an L4 this works; the first tick is slow
> due to op compilation. If ops fail to build, see the troubleshooting notes in the
> setup cell.


## 1. Configuration

In [ ]:

# =====================================================================
# CONFIGURATION
# =====================================================================
CONFIG = {
    "model_name": "StyleGAN2_ADA",
    "seed": 42,
    "image_size": 128,
    "batch_size": 32,           # StyleGAN2-ADA global batch (L4-friendly at 128)
    "kimg": 3000,               # training length in thousands of images (configurable)
    "snap": 10,                 # snapshot/metric interval (ticks)
    "gpus": 1,
    "cfg": "auto",              # StyleGAN2-ADA config preset
    "aug": "ada",               # adaptive discriminator augmentation (the "ADA")
    "mirror": 0,                # NO horizontal-flip augmentation (motif semantics)
    "metrics": "fid50k_full",   # native metric (reals = training set)
    "eval_n_gen": 640,          # for OUR held-out FID/KID recomputation
    "eval_kid_subset": 100,
    "conditional": False,       # baseline is unconditional (cWGAN-GP covers conditional)
    "resume": True,             # native resume from latest snapshot
    "stylegan_commit": "6f160b3d22b8b178ebe533a50d4d5e63aedba21d",
}


## 2. Colab setup (clone repos + install deps)

In [ ]:

# =====================================================================
# COLAB SETUP: clone our repo (src/ + splits) + official StyleGAN2-ADA + deps
# =====================================================================
import os, sys, subprocess, glob, json, time

REPO_URL = "https://github.com/sid-2k6/Textile_Pattern_GAN.git"
REPO_DIR = "/content/Textile_Pattern_GAN"
SG2_URL  = "https://github.com/NVlabs/stylegan2-ada-pytorch.git"
SG2_DIR  = "/content/stylegan2-ada-pytorch"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

if not os.path.isdir(os.path.join(SG2_DIR, ".git")):
    subprocess.run(["git", "clone", SG2_URL, SG2_DIR], check=True)
    subprocess.run(["git", "-C", SG2_DIR, "checkout", CONFIG["stylegan_commit"]], check=False)
sys.path.insert(0, SG2_DIR)

# StyleGAN2-ADA deps. Custom CUDA ops need ninja; click/psutil/scipy/requests used
# by train.py; torchmetrics/torch-fidelity for OUR comparable FID/KID.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja",
                "click", "psutil", "scipy", "requests", "imageio", "imageio-ffmpeg==0.4.3",
                "torchmetrics>=1.0.0", "torch-fidelity"], check=False)
print("Commit:", subprocess.run(["git", "-C", SG2_DIR, "rev-parse", "HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("Setup complete.")
print("Troubleshooting: if custom-op build fails, ensure a GPU runtime, restart "
      "runtime after installing ninja, and check that torch/CUDA match the repo "
      "expectations (see stylegan2-ada-pytorch/README).")


## 3. Environment verification

In [ ]:

# =====================================================================
# ENVIRONMENT VERIFICATION  (GPU / CUDA / PyTorch)
# =====================================================================
from batik_gan import env
ENV_INFO = env.print_environment()


## 4. Imports + reproducibility

In [ ]:

# =====================================================================
# IMPORTS + REPRODUCIBILITY
# =====================================================================
import numpy as np, pandas as pd, torch
from batik_gan import env, paths as P, data as D, metrics as M, viz, manifest as MAN
env.set_seed(CONFIG["seed"], deterministic=True)
DEVICE = env.get_device()


## 5. Google Drive

In [ ]:

# =====================================================================
# GOOGLE DRIVE MOUNT
# =====================================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print("Not running in Colab (or drive already mounted):", e)


## 6. Paths + validation

In [ ]:

# =====================================================================
# PATH CONFIGURATION  --- EDIT DATASET_ROOT TO MATCH YOUR DRIVE ---
# =====================================================================
DRIVE_ROOT    = "/content/drive/MyDrive"
DATASET_ROOT  = f"{DRIVE_ROOT}/Textile_Pattern_GAN/Datasets"          # <-- EDIT if needed
METADATA_PATH = f"{DATASET_ROOT}/Batik_Lasem/motifs (isen-isen)/metadata motifs.csv"
OUTPUT_ROOT   = f"{DRIVE_ROOT}/Textile_Pattern_GAN/outputs"
SPLITS_DIR    = os.path.join(REPO_DIR, "data", "splits")

paths = P.ProjectPaths(project_root=REPO_DIR, dataset_root=DATASET_ROOT,
                       metadata_path=METADATA_PATH, splits_dir=SPLITS_DIR,
                       output_root=OUTPUT_ROOT, model_name=CONFIG["model_name"]).make_dirs()
need_meta = not os.path.isfile(os.path.join(SPLITS_DIR, "batik_lasem_50pct_train.csv"))
P.validate_paths(paths, require_metadata=need_meta)

# StyleGAN2-ADA training-run dir (on Drive for resilience)
SG2_OUTDIR = os.path.join(OUTPUT_ROOT, "StyleGAN2_ADA", "training-runs")
SG2_DATA_ZIP = os.path.join(OUTPUT_ROOT, "StyleGAN2_ADA", "batik_train_128.zip")
os.makedirs(SG2_OUTDIR, exist_ok=True)


## 7. Load shared 50% split + audit

In [ ]:

# =====================================================================
# LOAD THE SHARED 50% SPLIT (same as DCGAN / cWGAN-GP)
# =====================================================================
train_csv = os.path.join(SPLITS_DIR, "batik_lasem_50pct_train.csv")
test_csv  = os.path.join(SPLITS_DIR, "batik_lasem_50pct_test.csv")
if not (os.path.isfile(train_csv) and os.path.isfile(test_csv)):
    MAN.build_all(METADATA_PATH, SPLITS_DIR, frac=0.5, test_frac=0.2, seed=CONFIG["seed"])
train_df = pd.read_csv(train_csv); test_df = pd.read_csv(test_csv)
AUDIT = D.audit_dataset(train_df, test_df, DATASET_ROOT, sample_check=400, stop_on_error=True)


## 8. Prepare StyleGAN2-ADA dataset (50% train, 128px)

In [ ]:

# =====================================================================
# PREPARE StyleGAN2-ADA DATASET FROM THE 50% TRAIN SPLIT
#   * identical preprocessing: bicubic+antialias resize to 128x128, RGB
#   * TEST split is NOT included (held out)
#   * dataset_tool.py packs the pre-resized PNGs into a zip (no extra resize)
# =====================================================================
from PIL import Image
from batik_gan.paths import resolve_image_path

STAGE_DIR = "/content/sg2_stage_train"
os.makedirs(STAGE_DIR, exist_ok=True)
if not os.path.isfile(SG2_DATA_ZIP):
    print("Staging 128x128 training PNGs ...")
    n_ok = 0
    for i, r in train_df.reset_index(drop=True).iterrows():
        ap = resolve_image_path(DATASET_ROOT, r.get("rel_path", ""), r["filename"])
        if ap is None:
            continue
        img = Image.open(ap).convert("RGB").resize(
            (CONFIG["image_size"], CONFIG["image_size"]), Image.BICUBIC)
        img.save(os.path.join(STAGE_DIR, f"{i:05d}.png"))
        n_ok += 1
    print(f"Staged {n_ok} images -> {STAGE_DIR}")
    # pack with the official dataset_tool (resolution already matches)
    cmd = [sys.executable, os.path.join(SG2_DIR, "dataset_tool.py"),
           f"--source={STAGE_DIR}", f"--dest={SG2_DATA_ZIP}",
           f"--width={CONFIG['image_size']}", f"--height={CONFIG['image_size']}"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Dataset zip already exists:", SG2_DATA_ZIP)


## 9. Train StyleGAN2-ADA (resume-capable)

In [ ]:

# =====================================================================
# TRAIN StyleGAN2-ADA  (native checkpoint/resume via --resume)
# =====================================================================
def latest_snapshot(outdir):
    runs = sorted(glob.glob(os.path.join(outdir, "*")))
    for run in reversed(runs):
        pkls = sorted(glob.glob(os.path.join(run, "network-snapshot-*.pkl")))
        if pkls:
            return pkls[-1]
    return None

resume_arg = []
prev = latest_snapshot(SG2_OUTDIR)
if CONFIG["resume"] and prev:
    print(f"Checkpoint found. Resuming from {prev}")
    resume_arg = [f"--resume={prev}"]
else:
    print("No snapshot found. Starting StyleGAN2-ADA from scratch.")

cmd = [sys.executable, os.path.join(SG2_DIR, "train.py"),
       f"--outdir={SG2_OUTDIR}", f"--data={SG2_DATA_ZIP}",
       f"--gpus={CONFIG['gpus']}", f"--cfg={CONFIG['cfg']}",
       f"--aug={CONFIG['aug']}", f"--mirror={CONFIG['mirror']}",
       f"--kimg={CONFIG['kimg']}", f"--snap={CONFIG['snap']}",
       f"--metrics={CONFIG['metrics']}", f"--seed={CONFIG['seed']}",
       f"--batch={CONFIG['batch_size']}"] + resume_arg
print("Running:", " ".join(cmd))
# NOTE: this is a long-running call; Colab may disconnect — just re-run this cell
# to resume from the latest snapshot on Drive.
subprocess.run(cmd, check=False)


## 10. Parse logs -> history.csv

In [ ]:

# =====================================================================
# PARSE StyleGAN2-ADA LOGS -> history.csv  (adapt to the metrics IT exposes)
#   stats.jsonl : per-tick Loss/G, Loss/D, timing, resources
#   metric-fid50k_full.jsonl : native FID (reals = training set)
# We DO NOT invent discriminator accuracy (undefined here) — those columns are
# omitted rather than zero-filled.
# =====================================================================
def newest_run(outdir):
    runs = [d for d in sorted(glob.glob(os.path.join(outdir, "*"))) if os.path.isdir(d)]
    return runs[-1] if runs else None

run_dir = newest_run(SG2_OUTDIR)
rows = []
if run_dir:
    stats_path = os.path.join(run_dir, "stats.jsonl")
    fid_path = os.path.join(run_dir, "metric-fid50k_full.jsonl")
    native_fid = {}
    if os.path.isfile(fid_path):
        for line in open(fid_path):
            d = json.loads(line)
            native_fid[int(round(d["snapshot_pkl"].split("-")[-1].split(".")[0])
                              if False else d.get("kimg", 0))] = d["results"]["fid50k_full"]
    if os.path.isfile(stats_path):
        for line in open(stats_path):
            d = json.loads(line)
            def g(k):
                return d.get(k, {}).get("mean", float("nan")) if isinstance(d.get(k), dict) else d.get(k, float("nan"))
            kimg = g("Progress/kimg")
            rows.append({
                "epoch": g("Progress/tick"),          # tick as the x-axis unit
                "kimg": kimg,
                "generator_loss": g("Loss/G/loss"),
                "discriminator_loss": g("Loss/D/loss"),
                "native_fid50k_full": native_fid.get(int(kimg) if kimg == kimg else -1, float("nan")),
                "epoch_time": g("Timing/sec_per_tick"),
                "gpu_memory_mb": g("Resources/peak_gpu_mem_gb") * 1024
                                 if g("Resources/peak_gpu_mem_gb") == g("Resources/peak_gpu_mem_gb") else float("nan"),
            })
hist_df = pd.DataFrame(rows)
if len(hist_df):
    hist_df.to_csv(paths.history_csv, index=False)
    print(f"Wrote {paths.history_csv} ({len(hist_df)} ticks)")
else:
    print("No stats.jsonl yet — run the training cell first.")


## 11. Comparable held-out FID/KID (same evaluator as other baselines)

In [ ]:

# =====================================================================
# COMPARABLE FID/KID: generate from the trained network and evaluate against
# OUR held-out TEST set with the SAME evaluator as DCGAN / cWGAN-GP.
# =====================================================================
import pickle
sys.path.insert(0, SG2_DIR)
import dnnlib, legacy   # from the official repo

def load_generator(pkl_path):
    with dnnlib.util.open_url(pkl_path) as f:
        G = legacy.load_network_pkl(f)["G_ema"].to(DEVICE).eval()
    return G

pkl = latest_snapshot(SG2_OUTDIR) if 'latest_snapshot' in dir() else None
if pkl is None:
    import glob as _glob
    cand = sorted(_glob.glob(os.path.join(SG2_OUTDIR, "*", "network-snapshot-*.pkl")))
    pkl = cand[-1] if cand else None

final_metrics = {"model": "StyleGAN2_ADA"}
if pkl:
    print("Evaluating snapshot:", pkl)
    Gnet = load_generator(pkl)

    def sample_generator(n):
        z = torch.randn(n, Gnet.z_dim, device=DEVICE)
        c = None
        if Gnet.c_dim > 0:
            c = torch.zeros(n, Gnet.c_dim, device=DEVICE)
        img = Gnet(z, c, truncation_psi=1.0, noise_mode="const")  # [-1,1] float
        return img

    real_uint8 = M.build_real_uint8_from_loader(
        D.make_dataloader(D.BatikCanonicalDataset(test_df, DATASET_ROOT, CONFIG["image_size"],
                                                  conditional=False, verify=True),
                          CONFIG["batch_size"], shuffle=False, num_workers=2,
                          seed=CONFIG["seed"], drop_last=False))
    evaluator = M.GenerativeEvaluator(real_uint8, DEVICE, kid_subset_size=CONFIG["eval_kid_subset"])
    res = evaluator.evaluate(sample_generator, n_gen=CONFIG["eval_n_gen"],
                             batch=CONFIG["batch_size"], compute_diversity=True)
    with torch.no_grad():
        z = torch.randn(64, Gnet.z_dim, device=DEVICE)
        c = torch.zeros(64, Gnet.c_dim, device=DEVICE) if Gnet.c_dim > 0 else None
        grid = Gnet(z, c, truncation_psi=1.0, noise_mode="const")
    viz.save_sample_grid(grid, os.path.join(paths.samples_dir, "final_grid.png"),
                         title="StyleGAN2-ADA final")
    n_params = sum(p.numel() for p in Gnet.parameters())
    final_metrics.update({
        "snapshot": os.path.basename(pkl),
        "final_fid": res["fid"], "final_kid": res["kid_mean"], "final_kid_std": res["kid_std"],
        "final_diversity": res["diversity"],
        "held_out_fid_note": "recomputed vs held-out TEST (comparable to DCGAN/cWGAN-GP)",
        "native_metric": "fid50k_full logged by StyleGAN2-ADA uses TRAINING reals (see history.csv)",
        "generator_parameters": int(n_params),
        "n_real_test": int(real_uint8.shape[0]), "n_gen_eval": CONFIG["eval_n_gen"],
        "real_accuracy": "N/A (StyleGAN2-ADA uses logistic/ADA; no fixed accuracy)",
        "generator_accuracy": "N/A (not defined for GANs)",
    })
else:
    final_metrics["status"] = "no snapshot found — train first"
    print("No network snapshot found yet.")


## 12. Save final metrics + plots

In [ ]:

# =====================================================================
# SAVE final metrics + plots
# =====================================================================
# best/final FID from native log (lower better) for the record
if len(hist_df) and "native_fid50k_full" in hist_df and hist_df["native_fid50k_full"].notna().any():
    bi = hist_df["native_fid50k_full"].idxmin()
    final_metrics["best_native_fid50k_full"] = float(hist_df.loc[bi, "native_fid50k_full"])
    final_metrics["best_tick"] = int(hist_df.loc[bi, "epoch"])
if len(hist_df):
    final_metrics["training_time_sec"] = float(pd.to_numeric(hist_df["epoch_time"], errors="coerce").sum())
    if hist_df["gpu_memory_mb"].notna().any():
        final_metrics["peak_gpu_memory_mb"] = float(hist_df["gpu_memory_mb"].max())
    final_metrics["final_generator_loss"] = float(pd.to_numeric(hist_df["generator_loss"], errors="coerce").dropna().iloc[-1]) if hist_df["generator_loss"].notna().any() else float("nan")
    final_metrics["final_discriminator_loss"] = float(pd.to_numeric(hist_df["discriminator_loss"], errors="coerce").dropna().iloc[-1]) if hist_df["discriminator_loss"].notna().any() else float("nan")

with open(paths.final_metrics_json, "w") as f:
    json.dump(final_metrics, f, indent=2, default=str)
pd.DataFrame([final_metrics]).to_csv(paths.final_metrics_csv, index=False)

# plots adapted to StyleGAN2-ADA metrics
if len(hist_df):
    viz.line_plot(hist_df, "epoch", ["generator_loss"], "StyleGAN2-ADA: Generator Loss vs Tick",
                  "Generator Loss", os.path.join(paths.plots_dir, "generator_loss.png"))
    viz.line_plot(hist_df, "epoch", ["discriminator_loss"], "StyleGAN2-ADA: Discriminator Loss vs Tick",
                  "Discriminator Loss", os.path.join(paths.plots_dir, "discriminator_loss.png"))
    viz.line_plot(hist_df, "epoch", ["native_fid50k_full"], "StyleGAN2-ADA: native FID50k vs Tick",
                  "FID50k_full (train reals)", os.path.join(paths.plots_dir, "fid.png"))
    viz.line_plot(hist_df, "epoch", ["epoch_time"], "StyleGAN2-ADA: Time per Tick",
                  "sec/tick", os.path.join(paths.plots_dir, "epoch_time.png"))
print(json.dumps(final_metrics, indent=2, default=str))


## 13. Final results summary

In [ ]:

# =====================================================================
# FINAL RESULTS SUMMARY
# =====================================================================
print("="*60); print("StyleGAN2-ADA — FINAL SUMMARY"); print("="*60)
for k in ("snapshot", "final_fid", "final_kid", "final_diversity",
          "best_native_fid50k_full", "best_tick", "generator_parameters",
          "training_time_sec", "peak_gpu_memory_mb"):
    if k in final_metrics:
        print(f"{k:26s}: {final_metrics[k]}")
print("Held-out FID/KID are comparable to DCGAN/cWGAN-GP (same evaluator + test set).")
print("Native fid50k_full (training reals) is StyleGAN2-ADA's own metric — do NOT")
print("compare it directly with the held-out numbers.")
print("="*60)
